## PydanticAI 제품 리뷰 구조화 분석

쿠팡에서 수집한 **선크림 3개 제품** 의 리뷰 데이터를 PydanticAI의 구조화된 출력으로 분석합니다.

| 학습 목표 | 내용 |
|---|---|
| 다중 제품 분석 | 3개 제품의 리뷰를 하나의 스키마로 통합 분석 |
| 범용 스키마 설계 | 특정 제품에 종속되지 않는 10가지 평가 항목 설계 |
| 층화 샘플링 | 별점별 균형 잡힌 리뷰 선정 |
| 파이프라인 구축 | 분석 결과를 JSON으로 저장하여 다음 노트북에서 활용 |

### 환경 설정

실습에 필요한 패키지를 불러오고, API 키를 설정합니다.

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
import json
import time
import glob
import platform
from pprint import pprint
from typing import List, Literal, Optional

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModelSettings

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')

# PydanticAI는 GEMINI_API_KEY 환경변수를 자동으로 인식합니다
# 모델 ID 형식: 'google-gla:{모델명}'
model_id = f'google-gla:{gemini_model}'

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'O' if api_key_valid else 'X'}")
if not api_key_valid:
    print("[주의] .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {model_id}")

# API 호출 간격 (초) - 무료 플랜에 맞게 조정
API_DELAY = 3

### 1. 제품 리뷰 데이터 로드

**실습 (1)**
- 선크림 3개 제품의 쿠팡 리뷰 JSON 데이터를 로드하고, 별점 분포와 리뷰 길이 등 기본 통계를 확인합니다.
- **실제 별점 분포에 비례** 하여 제품당 10건(총 30건)을 층화 샘플링합니다.

**층화 샘플링(Stratified Sampling)이란?**

전체 데이터를 특정 기준(여기서는 별점)으로 그룹(층)을 나눈 뒤, 각 그룹에서 **원래 비율에 맞게** 샘플을 추출하는 방법입니다.

| 방법 | 예시 (5점 96%, 3점 1%) | 문제점 |
|---|---|---|
| 단순 랜덤 샘플링 | 10건 중 5점만 10건 | 1~3점 리뷰가 누락될 수 있음 |
| 균등 샘플링 | 각 별점 2건씩 | 5점 비율이 과소 대표됨 |
| **층화 샘플링** | 5점 10건, 3점 1건 (비율 유지) | 원래 분포를 보존하면서 소수 의견도 포함 |

층화 샘플링을 사용하면 다수 의견(5점)의 비중을 유지하면서도, 소수 의견(1~2점)이 최소 1건은 포함되어 편향 없는 분석이 가능합니다.

여기서는 `sklearn`의 `StratifiedShuffleSplit`을 사용합니다. 비율 유지 + 정확한 N건 추출을 자동으로 처리해줍니다.

| 파라미터 | 값 | 설명 |
|---|---|---|
| `n_splits` | 1 | 샘플링을 1번만 수행합니다 |
| `train_size` | 10 | 추출할 샘플 수 (제품당 10건) |
| `random_state` | 42 | 재현 가능한 결과를 위한 시드 값 |

In [ ]:
# =============================================================================
#  제품 리뷰 JSON 데이터 로드
# =============================================================================

json_files = sorted(glob.glob("./data/product_*_reviews.json"))

if not json_files:
    print("[오류] data/product_*_reviews.json 파일을 찾을 수 없습니다.")
else:
    all_reviews = []
    product_info = {}

    for file_path in json_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        product_id = data['product_id']
        reviews = data['reviews']

        if reviews:
            product_name = reviews[0]['product_name']
            product_info[product_id] = {
                'name': product_name,
                'vendor': reviews[0]['vendor_name'],
                'total_reviews': data['total_reviews'],
                'loaded_reviews': len(reviews)
            }

        for review in reviews:
            all_reviews.append(review)

    # DataFrame 생성
    reviews_df = pd.DataFrame(all_reviews)

    print(f"제품 수: {len(product_info)}개")
    print(f"전체 리뷰 수: {len(reviews_df):,}개")
    print()

    # 제품별 통계
    print("제품별 리뷰 통계:")
    for pid, info in product_info.items():
        avg_rating = reviews_df[reviews_df['product_id'] == pid]['rating'].mean()
        print(f"  [{pid}] {info['name'][:40]}...")
        print(f"    리뷰 수: {info['loaded_reviews']}개 / 평균 평점: {avg_rating:.2f}점")

    # 데이터 전처리: 빈 리뷰 제거
    initial_count = len(reviews_df)
    reviews_df = reviews_df.dropna(subset=['review_content'])
    reviews_df = reviews_df[reviews_df['review_content'].str.strip() != '']
    reviews_df['review_length'] = reviews_df['review_content'].str.len()
    reviews_df = reviews_df[reviews_df['review_length'] > 10]  # 10자 이상만
    reviews_df = reviews_df.reset_index(drop=True)
    print(f"\n전처리 후: {len(reviews_df)}개 ({initial_count - len(reviews_df)}개 제거)")

In [ ]:
# =============================================================================
#  제품별 층화 샘플링 (제품당 100건, 실제 별점 분포에 비례)
#  - 층화 샘플링(Stratified Sampling)은 모집단을 특정 기준(층)으로 나눈 뒤, 각 층의 비율에 맞게 표본을 추출하는 방법입니다.
#    - 설문·여론조사에서 성별·연령·지역 비율을 반영해서 소수 집단이 누락되지 않고, 표본이 모집단을 대표하고 싶을 때 사용
#  - StratifiedShuffleSplit: 별점 비율을 유지하면서 정확히 N건을 추출합니다
# =============================================================================

SAMPLE_PER_PRODUCT = 100

sss = StratifiedShuffleSplit(
    n_splits=1,              # 1번만 분할, n_splits=3이면 서로 다른 랜덤 조합으로 3세트를 만듬
    train_size=SAMPLE_PER_PRODUCT,  # 추출할 샘플 수
    random_state=42          # 재현성을 위한 시드
)

sampled = []
for pid in product_info.keys():
    product_reviews = reviews_df[reviews_df['product_id'] == pid]

    print(f"[{pid}] 전체 {len(product_reviews)}개 => {SAMPLE_PER_PRODUCT}개 샘플링")

    # 층화 샘플링 실행 (별점을 기준으로 비율 유지)
    for sample_idx, _ in sss.split(product_reviews, product_reviews['rating']):
        product_sample = product_reviews.iloc[sample_idx]
        sampled.append(product_sample)

    # 샘플링된 별점 분포 확인
    sample_dist = product_sample['rating'].value_counts().sort_index()
    for rating, count in sample_dist.items():
        print(f"  {rating}점: {count}개")

df_sample = pd.concat(sampled).reset_index(drop=True)

print(f"\n분석 대상: {len(df_sample)}개 (제품별 층화 샘플링)")
for pid in product_info.keys():
    cnt = len(df_sample[df_sample['product_id'] == pid])
    print(f"  [{pid}] {cnt}개")

### 2. 제품 리뷰 분석 스키마 정의

**스키마 설계 프로세스**

기본_3에서는 단일 제품(자세 교정 밴드)에 특화된 9개 항목을 사용했습니다. 이번에는 **선크림 3개 제품** 을 하나의 스키마로 분석해야 하므로, 어떤 제품에도 적용 가능한 **범용 10가지 항목** 으로 확장합니다.

| 설계 단계 | 내용 |
|---|---|
| 1. 비즈니스 질문 | 3개 선크림 제품의 강점/약점을 항목별로 비교하려면 어떤 정보가 필요한가? |
| 2. 필요한 정보 | 전체 감정, 항목별 의견, 핵심 키워드, 장단점, 요약, 추천도 |
| 3. 스키마 구조 | 중첩 구조 (`AspectAnalysis` => `ProductReviewAnalysis`) |
| 4. 제약조건 | `Literal`로 항목/감정 범주 고정, `max_length`로 요약 길이 제한 |

**기본_3 대비 확장 포인트**

| 항목 | 기본_3 | 실전_1_1 | 이유 |
|---|---|---|---|
| 평가 항목 | 9개 (자세밴드 특화) | 10개 (범용) | 여러 제품 카테고리에 재사용 가능 |
| 감정 분류 | 3단계 (긍정/부정/중립) | 항목: 3단계, 전체: 4단계(+혼합) | 장단점 공존 리뷰를 "혼합"으로 구분 |
| 추천 여부 | `Optional[bool]` | `float (0.0~1.0)` | 3개 제품 순위 비교에 연속 값이 필요 |
| 장단점 | 없음 | `pros`, `cons` 리스트 | 비교 분석 시 워드클라우드 등에 활용 |

**실습 (2)**
- 범용 제품 리뷰 분석 스키마를 정의합니다.
- `AspectAnalysis`로 항목별 의견을 분리하고, `ProductReviewAnalysis`에서 전체 감정과 요약을 종합합니다.
- `Field(description=...)`에 분류 기준과 예시를 상세하게 기술하여, 시스템 프롬프트 없이도 LLM이 정확하게 분류하도록 유도합니다.

In [ ]:
# =============================================================================
#  제품 리뷰 분석 스키마 정의
# =============================================================================

# 리뷰에서 언급된 개별 평가 항목 (중첩 구조)
class AspectAnalysis(BaseModel):
    """리뷰에서 언급된 개별 평가 항목"""
    aspect: Literal[
        "품질", "사용성", "기능성", "디자인", "가성비",
        "내구성", "고객서비스", "배송", "포장", "전반적"
    ] = Field(
        description=(
            "리뷰에서 평가한 제품 관련 항목. "
            "품질: 제품의 만듦새, 재질, 성분, 제형 질감, 완성도. "
            "사용성: 사용 편의성, 바르는 느낌, 자극 여부, 흡수력, 직관적 사용법. "
            "기능성: 제품 본연의 효과/성능 달성도 (자외선 차단, 보습, 톤업 등). "
            "디자인: 외관, 용기 디자인, 색상, 미적 만족도, 휴대성. "
            "가성비: 가격 대비 만족도, 용량 대비 가격, 경제성. "
            "내구성: 지속력, 밀림 여부, 땀/물에 대한 유지력, 장기 사용 가능성. "
            "고객서비스: A/S, 문의 응답, 판매자 대응. "
            "배송: 배송 속도, 배송 상태, 로켓배송 만족도. "
            "포장: 제품 포장 상태, 파손 여부, 개봉 경험. "
            "전반적: 위 항목에 해당하지 않는 전반적 만족도, 재구매 의향"
        )
    )
    sentiment: Literal["긍정", "부정", "중립"] = Field(
        description=(
            "해당 항목에 대한 리뷰어의 감정. "
            "긍정: 만족, 추천, 칭찬 표현 ('좋아요', '편해요', '효과있어요' 등). "
            "부정: 불만, 실망, 비판 표현 ('불편해요', '별로', '아쉬워요' 등). "
            "중립: 단순 사실 언급이거나 평가가 모호한 경우 ('보통이에요', '그냥 그래요')"
        )
    )
    keywords: List[str] = Field(
        description="해당 항목 관련 핵심 키워드 (명사/형용사 위주, 리뷰 원문에서 추출)",
        min_length=1,
        max_length=3
    )
    detail: str = Field(
        description="해당 항목에 대한 리뷰어의 구체적 의견을 1문장으로 요약 (원문의 핵심 표현을 최대한 보존)",
        max_length=200
    )


# 메인 스키마: 제품 리뷰 분석 결과
class ProductReviewAnalysis(BaseModel):
    """제품 리뷰 구조화 분석 결과"""
    overall_sentiment: Literal["긍정", "부정", "중립", "혼합"] = Field(
        description=(
            "리뷰 전체의 감정 판정. "
            "긍정: 전반적으로 만족, 추천, 재구매 의사가 드러나는 리뷰. "
            "부정: 전반적으로 불만, 실망, 비추천이 드러나는 리뷰. "
            "중립: 감정 표현이 약하거나 단순 사실 나열에 그치는 리뷰. "
            "혼합: 장점과 단점이 뚜렷하게 공존하여 한쪽으로 판정하기 어려운 리뷰"
        )
    )
    aspects: List[AspectAnalysis] = Field(
        description="리뷰에서 실제로 언급된 평가 항목만 포함 (언급되지 않은 항목은 제외)",
        min_length=1,
        max_length=8
    )
    pros: List[str] = Field(
        description="장점 요약 (리뷰에서 긍정적으로 언급된 내용을 짧은 문구로 정리)",
        max_length=5
    )
    cons: List[str] = Field(
        description="단점 요약 (리뷰에서 부정적으로 언급된 내용을 짧은 문구로 정리)",
        max_length=5
    )
    summary: str = Field(
        description="리뷰 핵심 요약 1~2문장 (핵심 측면과 종합 의견 포함)",
        max_length=300
    )
    recommendation_score: float = Field(
        ge=0.0, le=1.0,
        description=(
            "추천도 점수 (0.0~1.0). "
            "0.8~1.0: 강력 추천. 0.6~0.8: 추천. 0.4~0.6: 보통. "
            "0.2~0.4: 비추천. 0.0~0.2: 강력 비추천. "
            "별점이 아닌 리뷰 텍스트 내용을 기준으로 판단"
        )
    )


print("제품 리뷰 분석 스키마 정의 완료")
print()

# BaseModel이 생성하는 JSON Schema 확인
print("=" * 60)
print("[ProductReviewAnalysis JSON Schema]")
print("=" * 60)
pprint(ProductReviewAnalysis.model_json_schema())

### 3. AI 리뷰 분석 실행

**실습 (4)**
- `Agent(output_type=ProductReviewAnalysis)`로 리뷰를 구조화된 형태로 분석합니다.
- 시스템 프롬프트에는 스키마 `description`과 중복되지 않는 **행동 지시만** 작성합니다.
- 원본 정보(`review_id`, `product_id`, `rating` 등)는 LLM 출력에 포함하지 않고 코드에서 합칩니다 (출력 토큰 절약).

In [ ]:
# =============================================================================
#  시스템 프롬프트 정의
#  - 분류 기준과 항목별 설명은 스키마 description에 포함되어 있으므로
#    여기서는 스키마로 표현할 수 없는 행동 지시만 작성합니다
# =============================================================================

review_system_prompt = """
당신은 제품 리뷰 분석 전문가입니다.
리뷰 텍스트를 읽고 구조화된 형태로 분석하세요.

[분석 원칙]
- 별점과 무관하게 리뷰 텍스트 자체의 감정과 내용을 기준으로 분석하세요.
- 한국어 구어체(ㅠㅠ, ㅋㅋ, ~요, 이모지)의 뉘앙스를 정확히 반영하세요.
- 내용이 없거나 의미 파악이 불가능한 리뷰는 overall_sentiment를 "중립"으로 처리하세요.
- 리뷰 원문에서 사용된 단어와 표현을 최대한 보존하세요.
- 언급되지 않은 항목은 제외하고, 실제로 언급된 항목만 분석하세요.
- 평점-텍스트 내용이 불일치하면 텍스트를 우선하세요.
"""

# Agent 생성 - output_type으로 구조화된 출력 강제
review_agent = Agent(
    model_id,
    output_type=ProductReviewAnalysis,
    system_prompt=review_system_prompt,
)

# 매개변수 설정 - 낮은 temperature로 일관된 분류 결과 유도
review_settings = GoogleModelSettings(temperature=0.2)

print("Agent 생성 완료")
print(f"  모델: {model_id}")
print(f"  출력 스키마: ProductReviewAnalysis")

In [ ]:
# =============================================================================
#  리뷰 분석 실행 + 비용 추정
#  - 10건 처리 후: 평균 토큰 사용량으로 전체 예상 비용 출력
#  - 전체 처리 후: 실제 누적 토큰 기반 최종 비용 출력
#  - 체크포인트: 5건마다 중간 저장 → 중단 후 재실행 시 이어서 처리
# =============================================================================
from pathlib import Path

# gemini-3.1-flash-lite 가격 (2026-03-23 기준, 백만 토큰당 USD)
INPUT_PRICE = 0.25
OUTPUT_PRICE = 0.50
EXCHANGE_RATE = 1500  # 원/달러 환율

CHECKPOINT_PATH = Path("outputs/_checkpoint_reviews.json")
CHECKPOINT_PATH.parent.mkdir(exist_ok=True)
CHECKPOINT_EVERY = 5  # N건마다 중간 저장

def save_checkpoint(results, path=CHECKPOINT_PATH):
    """중간 결과를 JSON 파일로 저장합니다"""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2, default=str)

def load_checkpoint(path=CHECKPOINT_PATH):
    """이전 중간 저장 결과를 복원합니다"""
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            results = json.load(f)
        print(f"체크포인트 복원: {len(results)}건 (이미 처리된 리뷰를 건너뜁니다)")
        return results
    return []

def print_cost_report(title, input_tokens, output_tokens, n_reviews=None, total_reviews=None):
    """토큰 사용량과 비용을 출력하는 헬퍼 함수입니다"""
    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE
    total_cost = input_cost + output_cost
    total_krw = total_cost * EXCHANGE_RATE

    print()
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)
    if n_reviews and total_reviews:
        print(f"  기준: {n_reviews}건 평균 => {total_reviews}건 전체 추정")
    print(f"  입력 토큰: {input_tokens:>10,} tokens  ()")
    print(f"  출력 토큰: {output_tokens:>10,} tokens  ()")
    print(f"  총 비용:     (₩{total_krw:.4f})")
    print("=" * 60)

print("제품 리뷰 구조화 분석 시작...")
print("=" * 80)

# 체크포인트 복원 (이전 중단 시점부터 이어서 처리)
analyzed_reviews = load_checkpoint()
processed_ids = {r['review_id'] for r in analyzed_reviews}

if processed_ids:
    print(f"스킵: {len(processed_ids)}건 (이미 처리됨)")

# 아직 처리하지 않은 리뷰만 필터링
pending_rows = [
    (idx, row) for idx, row in df_sample.iterrows()
    if str(row['review_id']) not in processed_ids
]

print(f"처리 대상: {len(pending_rows)}건")
print()

# 토큰 사용량 누적 추적
total_input_tokens = 0
total_output_tokens = 0
COST_SAMPLE_SIZE = 10  # 비용 추정에 사용할 샘플 수
new_count = 0  # 이번 실행에서 새로 처리한 건수

# 각 리뷰를 순회하며 PydanticAI Agent로 분석
for i, (_, row) in enumerate(pending_rows, 1):
    print(f"[리뷰 {len(analyzed_reviews)+1}/{len(df_sample)}] ({row['rating']}점) {row['review_content'][:50]}...")

    # LLM에 전달할 프롬프트 구성
    prompt = f"다음 제품 리뷰를 분석해주세요:

{row['review_content']}"

    try:
        # PydanticAI Agent 실행 (비동기)
        result = await review_agent.run(prompt, model_settings=review_settings)

        # 토큰 사용량 누적
        usage = result.usage()
        input_tok = usage.input_tokens or 0
        output_tok = usage.output_tokens or 0
        total_input_tokens += input_tok
        total_output_tokens += output_tok

        # result.output => ProductReviewAnalysis 객체 (자동 파싱)
        output = result.output

        # 딕셔너리로 변환 + 원본 정보는 코드에서 합치기 (출력 토큰 절약)
        analyzed = {
            "review_id": str(row["review_id"]),
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "rating": row["rating"],
            "review_content": row["review_content"],
            **output.model_dump()
        }
        analyzed_reviews.append(analyzed)
        new_count += 1

        # 분석 결과 출력
        print(f"   감정: {output.overall_sentiment} | 추천도: {output.recommendation_score:.2f}")
        aspects_str = ", ".join(f"{a.aspect}({a.sentiment})" for a in output.aspects)
        print(f"   항목: {aspects_str}")

    except Exception as e:
        print(f"   [실패] {str(e)[:100]}")

    # --- 10건 처리 후 예상 비용 출력 ---
    if new_count == COST_SAMPLE_SIZE and new_count > 0:
        avg_input = total_input_tokens / new_count
        avg_output = total_output_tokens / new_count
        est_total_input = int(avg_input * len(df_sample))
        est_total_output = int(avg_output * len(df_sample))
        print_cost_report(
            f"예상 비용 ({COST_SAMPLE_SIZE}건 평균 기반 추정)",
            est_total_input, est_total_output,
            n_reviews=new_count, total_reviews=len(df_sample)
        )
        print(f"
  1건 평균: 입력 {avg_input:.0f} / 출력 {avg_output:.0f} tokens")
        print()

    # N건마다 중간 저장 (커널 중단 대비)
    if new_count % CHECKPOINT_EVERY == 0 and new_count > 0:
        save_checkpoint(analyzed_reviews)

    time.sleep(API_DELAY)

# 최종 체크포인트 저장
save_checkpoint(analyzed_reviews)

# 전체 처리 결과 요약
print()
print("=" * 80)
print(f"총 {len(analyzed_reviews)}개 리뷰 분석 완료 (이번 실행: {new_count}건)")

# --- 전체 처리 후 실제 비용 출력 ---
if new_count > 0:
    print_cost_report(
        "실제 비용 (이번 실행)",
        total_input_tokens, total_output_tokens
    )
    print(f"  1건 평균: 입력 {total_input_tokens / new_count:.0f} / 출력 {total_output_tokens / new_count:.0f} tokens")

### 3-1. (도전) 대용량 배치 처리 패턴

위의 단순 루프는 학습용으로 충분하지만, 실무에서 수백~수천 건의 리뷰를 처리할 때는 다음과 같은 문제가 발생합니다.

| 문제 | 원인 | 해결 패턴 |
|---|---|---|
| 처리 속도 | 1건씩 순차 호출 | `asyncio.gather`로 N건 동시 호출 (비동기 배치) |
| API Rate Limit | 분당/일일 호출 제한 초과 | 지수 백오프(Exponential Backoff) 재시도 |
| 중간 실패 | 네트워크 오류, 파싱 실패 | 예외별 분기 처리 + 중간 저장 |
| 진행률 파악 불가 | 긴 처리 시간 동안 상태 모름 | tqdm 진행률 바 + 실시간 통계 |
| 재시작 시 처음부터 | 중단 후 이미 처리한 건 재처리 | 체크포인트 저장/복원 |
| 비용 낭비 | 불필요하게 긴 입력, 중복 호출 | 토큰 추정 + 이미 처리된 건 스킵 |

**비동기 배치 처리 흐름**

```
[리뷰1, 리뷰2] ──gather──> [API 호출1, API 호출2] ──동시완료──> [결과1, 결과2]
      ↓ 대기(BASE_DELAY)
[리뷰3, 리뷰4] ──gather──> [API 호출3, API 호출4] ──동시완료──> [결과3, 결과4]
      ↓ ...
```

`BATCH_SIZE=2`이면 2건씩 동시 호출합니다. Rate Limit에 맞춰 조절할 수 있습니다.

**실습 (도전)**
- 아래 코드는 위의 단순 루프와 **동일한 결과** 를 내면서, 실무 환경에서 안정적으로 동작하는 패턴을 보여줍니다.
- `analyzed_reviews` 변수를 덮어쓰므로, 이 셀을 실행하면 위의 단순 루프 결과가 대체됩니다.

In [ ]:
# =============================================================================
#  [도전] 대용량 배치 처리 - 실전 패턴
#  - 비동기 배치(asyncio.gather), 지수 백오프, 중간 저장, 진행률, 비용 추정
#  - 10건 처리 후 평균 기반 예상 비용 + 전체 처리 후 실제 비용
# =============================================================================

import asyncio
from pathlib import Path
from tqdm.notebook import tqdm

# -------------------------------------------------------------------------
#  1. 설정값
# -------------------------------------------------------------------------
BATCH_SIZE = 2             # 동시에 API 호출할 건수 (Rate Limit에 맞게 조절)
CHECKPOINT_PATH = Path("outputs/_checkpoint_reviews.json")  # 중간 저장 파일
CHECKPOINT_PATH.parent.mkdir(exist_ok=True)
CHECKPOINT_EVERY = 5       # N건마다 중간 저장
MAX_RETRIES = 3            # 최대 재시도 횟수
BASE_DELAY = 3             # 배치 간 대기 시간 (초)
MAX_DELAY = 60             # 최대 대기 시간 (초)
COST_SAMPLE_SIZE = 10      # 비용 추정에 사용할 샘플 수

# gemini-3.1-flash-lite 가격 (2026-03-23 기준, 백만 토큰당 USD)
INPUT_PRICE = 0.25
OUTPUT_PRICE = 0.50
EXCHANGE_RATE = 1500  # 원/달러 환율

# -------------------------------------------------------------------------
#  2. 체크포인트 저장/복원
# -------------------------------------------------------------------------
def save_checkpoint(results, path=CHECKPOINT_PATH):
    """중간 결과를 JSON 파일로 저장합니다"""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2, default=str)

def load_checkpoint(path=CHECKPOINT_PATH):
    """이전 중간 저장 결과를 복원합니다"""
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            results = json.load(f)
        print(f"체크포인트 복원: {len(results)}건 (이미 처리된 리뷰를 건너뜁니다)")
        return results
    return []

# -------------------------------------------------------------------------
#  3. 지수 백오프 재시도 함수 (개별 리뷰 1건 처리)
# -------------------------------------------------------------------------
async def analyze_one(agent, row, settings, max_retries=MAX_RETRIES):
    """
    리뷰 1건을 분석합니다. 실패 시 지수 백오프로 재시도합니다.

    지수 백오프(Exponential Backoff):
      재시도할 때마다 대기 시간을 2배씩 늘리는 전략입니다.
      대기 시간 = BASE_DELAY * (2 ** 시도 횟수)
      예) BASE_DELAY=3 => 1회차: 3초, 2회차: 6초, 3회차: 12초
      => 서버가 과부하일 때 즉시 재시도하면 상황이 악화되므로,
         점점 더 오래 기다려서 서버가 회복할 시간을 줍니다.
    """
    review_id = str(row['review_id'])
    prompt = f"다음 제품 리뷰를 분석해주세요:\n\n{row['review_content']}"

    for attempt in range(max_retries):
        try:
            result = await agent.run(prompt, model_settings=settings)

            # 토큰 사용량 추출
            usage = result.usage()
            input_tok = usage.input_tokens or 0
            output_tok = usage.output_tokens or 0

            # 성공: 원본 정보 + LLM 출력 합치기
            return {
                "review_id": review_id,
                "product_id": row["product_id"],
                "product_name": row["product_name"],
                "rating": row["rating"],
                "review_content": row["review_content"],
                **result.output.model_dump()
            }, None, (input_tok, output_tok)

        except Exception as e:
            error_msg = str(e)
            is_rate_limit = '429' in error_msg or 'rate' in error_msg.lower()

            if attempt < max_retries - 1:
                delay = min(BASE_DELAY * (2 ** attempt), MAX_DELAY)
                if is_rate_limit:
                    delay = min(delay * 2, MAX_DELAY)
                print(f"     [{review_id}] 재시도 {attempt + 1}/{max_retries} ({delay}초 대기)")
                await asyncio.sleep(delay)
            else:
                return None, f"[{review_id}] {error_msg[:100]}", (0, 0)

    return None, f"[{review_id}] 알 수 없는 오류", (0, 0)

# -------------------------------------------------------------------------
#  4. 메인 배치 처리
# -------------------------------------------------------------------------

# 체크포인트 복원 (이전 중단 시점부터 이어서 처리)
analyzed_reviews = load_checkpoint()
processed_ids = {r['review_id'] for r in analyzed_reviews}

# 아직 처리하지 않은 리뷰만 필터링
pending_rows = [
    row for _, row in df_sample.iterrows()
    if str(row['review_id']) not in processed_ids
]

if processed_ids:
    print(f"스킵: {len(processed_ids)}건 (이미 처리됨)")

print(f"처리 대상: {len(pending_rows)}건 (배치 크기: {BATCH_SIZE})")
print()

# 처리 통계 및 토큰 추적
stats = {'success': len(analyzed_reviews), 'fail': 0}
total_input_tokens = 0
total_output_tokens = 0
cost_estimated = False  # 예상 비용 출력 여부

# BATCH_SIZE건씩 묶어서 동시 호출
pbar = tqdm(total=len(pending_rows), desc="리뷰 분석", initial=0)

for batch_start in range(0, len(pending_rows), BATCH_SIZE):
    batch = pending_rows[batch_start:batch_start + BATCH_SIZE]

    # asyncio.gather: 배치 내 리뷰들을 동시에 API 호출
    tasks = [analyze_one(review_agent, row, review_settings) for row in batch]
    results = await asyncio.gather(*tasks)

    # 결과 수집
    for analyzed, error, (in_tok, out_tok) in results:
        total_input_tokens += in_tok
        total_output_tokens += out_tok

        if analyzed:
            analyzed_reviews.append(analyzed)
            stats['success'] += 1
        else:
            print(f"  [최종 실패] {error}")
            stats['fail'] += 1

    pbar.update(len(batch))
    pbar.set_postfix(성공=stats['success'], 실패=stats['fail'])

    # N건마다 중간 저장 (커널 중단 대비)
    if stats['success'] % CHECKPOINT_EVERY == 0 and stats['success'] > 0:
        save_checkpoint(analyzed_reviews)

    # 배치 간 대기 (Rate Limit 방지)
    await asyncio.sleep(BASE_DELAY)

pbar.close()

# 최종 저장
save_checkpoint(analyzed_reviews)

# -------------------------------------------------------------------------
#  6. 처리 결과 요약 + 실제 비용
# -------------------------------------------------------------------------
print()
print("=" * 60)
print(f"배치 처리 완료 (BATCH_SIZE={BATCH_SIZE})")
print(f"  성공: {stats['success']}건")
print(f"  실패: {stats['fail']}건")
print(f"  성공률: {stats['success'] / max(stats['success'] + stats['fail'], 1) * 100:.1f}%")
print(f"  체크포인트: {CHECKPOINT_PATH}")
print("=" * 60)

# --- 전체 처리 후 실제 비용 출력 ---
print_cost_report(
    "실제 비용 (전체 처리 완료)",
    total_input_tokens, total_output_tokens
)
new_success = stats['success'] - len(processed_ids)
if new_success > 0:
    print(f"  1건 평균: 입력 {total_input_tokens / new_success:.0f} / 출력 {total_output_tokens / new_success:.0f} tokens")

### 4. 분석 결과 확인

In [ ]:
# 분석 결과를 DataFrame으로 변환하여 표 형태로 확인합니다
df_analysis = pd.DataFrame(analyzed_reviews)

print("=== 제품 리뷰 분석 결과 ===")
print(f"총 {len(df_analysis)}개 리뷰 분석 완료")
print()

display(df_analysis.sample(10))

### 5. 분석 결과 시각화

**실습 (5)**
- 분석 결과를 시각화하여 리뷰 감정 분포, 별점별 감정 교차 분석, 평가 항목별 의견을 확인합니다.

In [ ]:
# aspects(중첩 리스트)를 플랫하게 펼칩니다
all_aspects = []
for row in analyzed_reviews:
    for asp in row['aspects']:
        all_aspects.append(asp)
df_aspects = pd.DataFrame(all_aspects)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) 리뷰 감정 분포
sentiment_order = ['긍정', '혼합', '중립', '부정']
sentiment_vals = [s for s in sentiment_order if s in df_analysis['overall_sentiment'].values]
sentiment_colors = {'긍정': '#4CAF50', '혼합': '#FFC107', '중립': '#9E9E9E', '부정': '#F44336'}
sns.countplot(
    data=df_analysis, x='overall_sentiment', order=sentiment_vals,
    palette=[sentiment_colors[s] for s in sentiment_vals], ax=axes[0]
)
axes[0].set_title('리뷰 감정 분포')
axes[0].set_xlabel('')
axes[0].set_ylabel('건수')

# 2) 별점별 감정 분포 (교차 분석)
sns.countplot(
    data=df_analysis, x='rating', hue='overall_sentiment',
    hue_order=sentiment_vals,
    palette=sentiment_colors, ax=axes[1]
)
axes[1].set_title('별점별 감정 분포')
axes[1].set_xlabel('별점')
axes[1].set_ylabel('건수')
axes[1].legend(title='감정', fontsize=8)

# 3) 평가 항목별 의견 분포
opinion_order = ['긍정', '중립', '부정']
opinion_vals = [o for o in opinion_order if o in df_aspects['sentiment'].values]
opinion_colors = {'긍정': '#4CAF50', '중립': '#9E9E9E', '부정': '#F44336'}
aspect_order = df_aspects['aspect'].value_counts().index.tolist()
sns.countplot(
    data=df_aspects, y='aspect', hue='sentiment',
    order=aspect_order, hue_order=opinion_vals,
    palette=opinion_colors, ax=axes[2]
)
axes[2].set_title('평가 항목별 의견 분포')
axes[2].set_xlabel('건수')
axes[2].set_ylabel('')
axes[2].legend(title='의견', fontsize=8)

plt.suptitle('제품 리뷰 구조화 분석 결과', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
#  레이더 차트: 평가 항목별 긍정/부정 비율
# =============================================================================
import numpy as np

# 항목별 긍정/부정/중립 건수 집계
aspect_stats = df_aspects.groupby(['aspect', 'sentiment']).size().unstack(fill_value=0)

# 전체 항목 목록 (데이터에 존재하는 것만)
aspects_list = aspect_stats.index.tolist()
n_aspects = len(aspects_list)

if n_aspects >= 3:  # 레이더 차트는 최소 3개 항목이 필요합니다
    # 긍정/부정 비율 계산
    total_per_aspect = aspect_stats.sum(axis=1)
    positive_ratio = (aspect_stats.get('긍정', 0) / total_per_aspect).values
    negative_ratio = (aspect_stats.get('부정', 0) / total_per_aspect).values

    # 레이더 차트 각도 계산
    angles = np.linspace(0, 2 * np.pi, n_aspects, endpoint=False).tolist()
    angles += angles[:1]  # 닫힌 다각형을 위해 첫 값 반복

    positive_ratio = np.append(positive_ratio, positive_ratio[0])
    negative_ratio = np.append(negative_ratio, negative_ratio[0])

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    # 긍정 비율 (초록)
    ax.fill(angles, positive_ratio, alpha=0.2, color='#4CAF50')
    ax.plot(angles, positive_ratio, color='#4CAF50', linewidth=2, label='긍정 비율')
    ax.scatter(angles[:-1], positive_ratio[:-1], color='#4CAF50', s=40, zorder=5)

    # 부정 비율 (빨강)
    ax.fill(angles, negative_ratio, alpha=0.2, color='#F44336')
    ax.plot(angles, negative_ratio, color='#F44336', linewidth=2, label='부정 비율')
    ax.scatter(angles[:-1], negative_ratio[:-1], color='#F44336', s=40, zorder=5)

    # 축 라벨 설정
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(aspects_list, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='grey')

    ax.set_title('평가 항목별 긍정/부정 비율 (레이더 차트)',
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print(f"레이더 차트를 그리려면 최소 3개 항목이 필요합니다 (현재: {n_aspects}개)")

### 결과 저장

**실습 (6)**
- 분석 결과를 JSON 파일로 저장합니다.
- 이 파일은 다음 노트북(실전_1_2)에서 제품 비교 분석의 입력 데이터로 사용됩니다.

In [ ]:
# =============================================================================
#  분석 결과 JSON 저장 (outputs 폴더)
# =============================================================================
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_filename = output_dir / "product_review_analysis.json"

# DataFrame을 JSON으로 저장
df_analysis.to_json(output_filename, orient='records', force_ascii=False, indent=2)

print(f"분석 결과 저장 완료")
print(f"  파일명: {output_filename}")
print(f"  저장된 리뷰 수: {len(df_analysis)}개")
print()
print("다음 단계: 실전_1_2_제품비교_리뷰분석.ipynb 에서 이 파일을 로드하여 제품 비교 분석을 진행합니다.")